<a href="https://colab.research.google.com/github/Manasa-Joshi24/AI_Travel_Concierge/blob/main/week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install -q langchain==1.0.5 langchain-community==0.4.1 langchain-groq==1.0.0 langchain-google-genai==3.0.1 chromadb==1.3.4 pypdf python-dotenv pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 359.4/359.4 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
google-adk 1.28.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.28.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
google-adk 1.28.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.


In [14]:
!pip install langchain-core langchain-community langchain-google-genai langchain-text-splitters chromadb -q

In [15]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
print("API key loaded ✅")

API key loaded ✅


In [16]:
import json

with open("/content/bengaluru_places_rag.json") as f:
  data = json.load(f)

print(f"✅ Loaded {len(data['places'])} places from JSON")

✅ Loaded 37 places from JSON


In [17]:
import json
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings


places = data["places"]

documents = []
for place in places:
    content = f"""
Name: {place['name']}
Address: {place['address']}
Rating: {place['rating']} ({place['rating_count']} reviews)
Hours: {place['hours']}
Budget: {place['budget']['range']} — {place['budget']['notes']}
Description: {place['description']}
Highlights: {', '.join(place['highlights'])}
Best for (who): {', '.join(place['tags']['who'])}
Best for (time): {', '.join(place['tags']['time'])}
Best for (vibe): {', '.join(place['tags']['vibe'])}
""".strip()

    documents.append(Document(
    page_content=content,
    metadata={
        "name": place["name"],
        "budget_level": place["budget"]["level"],
        "who": ", ".join(place["tags"]["who"]),
        "time": ", ".join(place["tags"]["time"]),
        "vibe": ", ".join(place["tags"]["vibe"]),
        "rating": place["rating"],
        "address": place["address"]
    }
))
print(f"✅ {len(documents)} places loaded as documents")

splitter = CharacterTextSplitter(chunk_size=600, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"✅ {len(chunks)} chunks created")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GEMINI_API_KEY)

vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ ChromaDB vector store ready — Lila's knowledge base is live")



✅ 37 places loaded as documents
✅ 37 chunks created
✅ ChromaDB vector store ready — Lila's knowledge base is live


In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

llm=ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0.3,
    max_tokens=5000
)

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are Lila, a warm and helpful travel and hangout companion for people in Bengaluru.
Based on the places information below, suggest the most relevant options.
For each place give the name, why it fits the mood, budget, and highlights.
Keep it friendly and concise.

Places information:
{context}

User's mood and request:
{question}

Your suggestions:
"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Lila's chain is ready")

✅ Lila's chain is ready


In [24]:
# Query 1
query = "I am going with friends, have a few hours, want a chill vibe, staying in Bengaluru"
response = qa_chain.invoke(query)
print("Query:", query)
print("Lila:", response)

# Query 2
query2 = "Solo trip, weekend, adventure mood, okay to go 100km outside Bengaluru"
response2 = qa_chain.invoke(query2)
print("\nQuery:", query2)
print("Lila:", response2)

# Query 3 — Priya's persona
query3 = "Chill spot tonight, not too expensive, near Indiranagar, group of 4"
response3 = qa_chain.invoke(query3)
print("\nQuery:", query3)
print("Lila:", response3)

Query: I am going with friends, have a few hours, want a chill vibe, staying in Bengaluru
Lila: **1️⃣ Elements Mall – Thanisandra (North Bengaluru)**  
- **Why it fits your vibe:** A calm, uncrowded spot perfect for a relaxed hang‑out with friends.  
- **Budget:** ₹200–₹800 per person (free entry).  
- **Highlights:**  
  - Cozy food court with a variety of quick bites.  
  - PVR cinema for a quick movie break.  
  - Supermarket inside for any last‑minute needs.  

**2️⃣ VR Bengaluru – Whitefield**  
- **Why it fits your vibe:** Spacious, quieter than the nearby Phoenix Marketcity, great for a chill afternoon.  
- **Budget:** ₹400–₹1200 per person (free entry; IMAX ₹300–450).  
- **Highlights:**  
  - PVR IMAX screen for an immersive movie experience.  
  - LuLu Hypermarket for shopping or snacks.  
  - Adventure climbing zone – a fun, low‑intensity activity if you feel like a quick stretch.  

Both spots are ideal for a few hours with friends, offering a relaxed atmosphere and plenty 